**Set the GPU runtime before running any cell.**

1. Runtime > Change runtime type.
2. Hardware accelerator: a GPU (T4 is free; L4, A100 and G4 are faster but use compute units). Click Save.
3. If Colab says the GPU is unavailable, pick another one. Do not connect without a GPU.

![Runtime menu, Change runtime type](https://raw.githubusercontent.com/Silent0Wings/ShapeSense/1337f82af9c096dcb13dbf318134a7012a8d5cd4/assets/ScreenShot/change%20to%20T4_Change%20runtime%20type.png)

![Change runtime type dialog with T4 GPU selected](https://raw.githubusercontent.com/Silent0Wings/ShapeSense/1337f82af9c096dcb13dbf318134a7012a8d5cd4/assets/ScreenShot/T4%20GPU.png)


**Disclaimer: this notebook has not been run end to end.** It has no saved outputs.

Known issues, found by a read-only audit on 2026-11-20:

- Run it on a GPU runtime (Runtime > Change runtime type > a GPU: T4, L4, A100 or G4). The cell that checks for CUDA stops the notebook on a CPU runtime.
- PyTorch prints `FutureWarning: torch.jit.script is deprecated` in Stage 4 and `torch.jit.load is deprecated` in Stage 5. They are expected and harmless: Stage 4 saves each model as a TorchScript file and Stages 5 and 6 load it back, which still works. The runner cell hides both; they only show in outputs from an older copy of this notebook.
- The install step prints pip dependency-conflict warnings for Colab's own packages (google-colab, cudf, numba, bigframes). They are expected and safe to ignore: the pipeline imports none of those packages. Do not downgrade pandas or numpy to silence them, because the pins match the recorded runs.
- The setup cell restarts the session once on purpose, to load the new packages. Older copies of this notebook did this by killing the process, which Colab reports as "Your session crashed". That message is expected too. After the restart, carry on with the next cell: every cell that needs the project folder moves into it by itself.
- Section 8 replays every raw image and retrains every row of the improvement table. It runs for a long time; stopping it leaves its later steps without inputs.
- Stage 0 is skipped on Colab: it requires Python 3.12 and Colab runs 3.13. The other stages run the same code through the Colab runner cell.
- The approximate times above each code cell come from the original machine (RTX 3080 laptop GPU) and from one Colab run of the setup cells. Colab's T4 GPU is slower, so the long stages take longer there.
- Run the cells in order from the top. Stages 2 to 6 depend on earlier outputs.
- The data comes from the GitHub release `data_set` in the data cell. No token is needed: the repo is public.
- The install step uses the default PyPI torch build, not the CUDA 13.0 build used for the recorded results, so numbers can differ.
- Cell 18 overwrites `results/improvement_table.json` and logs a test-split read in `artifacts/test_access.log`.
- The sealed fresh holdouts come from the release archive, so they are the recorded ones.

Recorded results are in `results/`. This notebook is a copy of the pipeline steps, not a verified re-run.

# ShapeSense: final pipeline

Runs top to bottom. One section per stage. The data (`data_set/`) is not in the repository; section 2 downloads it from the GitHub release.

Colab link: <https://colab.research.google.com/github/Silent0Wings/ShapeSense/blob/main/notebooks/shapesense_pipeline.ipynb>

## 0. Runtime check (needs a CUDA GPU)

**Why this cell requires a GPU.** The pipeline is set to `device: cuda` in `configs/pipeline.yaml`, and it stops when no GPU is visible rather than switching to CPU. A CPU run would not match the recorded GPU results. The stages load saved models with `map_location="cpu"`, so a model trained on CUDA loads on any machine. If you only want to run a finished model for inference, you can use `device = torch.device("cuda" if torch.cuda.is_available() else "cpu")` with `torch.load(path, map_location=device)` in a separate cell.

*Approximate time: under 10 seconds.*


In [ ]:
# Any NVIDIA GPU works (T4, L4, A100, G4). TPUs and the CPU do not.
import shutil, subprocess, torch

if not torch.cuda.is_available():
    gpu = ''
    if shutil.which('nvidia-smi'):
        gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip()
    if not gpu:
        raise RuntimeError(
            'This session has no GPU attached, even if one is selected. '
            'Runtime > Change runtime type > pick T4, L4, A100 or G4 > Save, and accept the restart. '
            'If Colab says that GPU is unavailable, do not connect without a GPU: pick another GPU or try later.')
    raise RuntimeError(
        f'A GPU is attached ({gpu}) but PyTorch {torch.__version__} (CUDA {torch.version.cuda}) cannot use it. '
        'Runtime > Restart session, then run this cell again.')

print(torch.cuda.get_device_name(0), '| PyTorch', torch.__version__, '| CUDA', torch.version.cuda)


## 1. Get the code and install pins

*Approximate time: about 1 minute for the clone and install; the session then restarts once.*


In [ ]:
# Clone the repo and install the pinned packages. Safe to run more than once.
# The pins upgrade numpy and pandas past Colab's built-in versions, which some preinstalled Colab packages
# (google-colab, numba, cudf, bigframes) do not accept. The pipeline imports none of them, so the pins stay:
# downgrading pandas or numpy would no longer match the recorded runs. pip's full output is kept quiet and
# shown only if the install fails. After the first install this cell restarts the session once.
import os, re, subprocess, sys

os.chdir('/content')
if not os.path.isdir('/content/ShapeSense'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/Silent0Wings/ShapeSense.git'], check=True)
os.chdir('/content/ShapeSense')

MARKER = '/content/.shapesense_installed'
if not os.path.exists(MARKER):
    pip = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'],
                         capture_output=True, text=True)
    if pip.returncode != 0:
        print(pip.stdout, pip.stderr)
        raise RuntimeError('pip install failed; see the output above')
    unused = sorted(set(re.findall(r'^(\S+) \S+ requires .* which is incompatible', pip.stderr, re.M)))
    if unused:
        print('Expected: these preinstalled Colab packages want older pins; the pipeline does not use them:',
              ', '.join(unused))
    open(MARKER, 'w').close()
    print('Packages installed. Restarting the session to load them; then continue with the next cell.')
    # A clean restart. Killing the process works too, but Colab then reports a crash.
    try:
        get_ipython().kernel.do_shutdown(True)
    except Exception:
        os.kill(os.getpid(), 9)
else:
    print('Repo and packages already in place:', os.getcwd())


After the restart (a "session restarted" or "session crashed" notice is expected here), Colab starts again in `/content`. The cells below move into the project folder by themselves, so just continue in order. The next cell only shows the working folder. Do not rerun the cells above it.

*Approximate time: instant.*


In [ ]:
import os
os.chdir('/content/ShapeSense')
print('Working folder:', os.getcwd())


## 2. Raw data

Download the data archive from the GitHub release `data_set` and unpack it into the clone. It holds the whole `data_set/` folder (329 MB, 214,857 files): the four raw sources, the built unified dataset that Stage 0 checks against, and the sealed fresh holdouts that Stage 5 reads. Details: `data/README.md`.

*Approximate time: about 30 seconds to 2 minutes, depending on download speed (329 MB archive).*


In [ ]:
# Download the whole data_set folder from the GitHub release and unpack it into the clone.
# Public repo: no token needed. Private repo: store a GitHub token as the Colab secret GITHUB_TOKEN.
import json, os, shutil, urllib.request, zipfile

if os.path.isdir('/content/ShapeSense'):  # Colab: a restart puts us back in /content
    os.chdir('/content/ShapeSense')

try:
    from google.colab import userdata
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    TOKEN = None

REPO = 'Silent0Wings/ShapeSense'

def gh(url, accept='application/vnd.github+json'):
    headers = {'Accept': accept}
    if TOKEN:
        headers['Authorization'] = f'Bearer {TOKEN}'
    return urllib.request.urlopen(urllib.request.Request(url, headers=headers))

NEEDED = ['2D geometric shapes dataset', 'kaggle_reevald', 'github_handdrawn', 'quickdraw',
          'unified', 'fresh_holdout', 'fresh_holdout_2']
if not all(os.path.isdir(f'data_set/{d}') for d in NEEDED):
    if TOKEN:
        release = json.load(gh(f'https://api.github.com/repos/{REPO}/releases/tags/data_set'))
        url = next(a['url'] for a in release['assets'] if a['name'] == 'shapesense_data_set_v1.zip')
        accept = 'application/octet-stream'
    else:
        url = f'https://github.com/{REPO}/releases/download/data_set/shapesense_data_set_v1.zip'
        accept = '*/*'
    with gh(url, accept=accept) as r, open('/content/data_set.zip', 'wb') as f:
        shutil.copyfileobj(r, f)
    zipfile.ZipFile('/content/data_set.zip').extractall('.')

print('data_set folders:', sorted(os.listdir('data_set')))


## 3. Guards and contract tests

*Approximate time: under 10 seconds.*


In [ ]:
import os
if os.path.isdir('/content/ShapeSense'):  # Colab: a restart puts us back in /content
    os.chdir('/content/ShapeSense')
!python run_pipeline.py --check

**Colab runner.** `run_pipeline.py --stages N` always runs Stage 0 first. Stage 0 checks the original machine: the Python 3.12 pin in `.python-version` and a rebuild of the recorded dataset. Colab runs Python 3.13, so Stage 0 cannot pass here, and no later stage reads its result. The cell below runs the same guards and the same stage code through `run_pipeline.py`'s own functions, without Stage 0. This is a recorded deviation from the local run. If a stage fails, the cell prints that stage's checks and error.

*Approximate time: instant.*


In [ ]:
import os, sys
if os.path.isdir('/content/ShapeSense'):  # Colab: a restart puts us back in /content
    os.chdir('/content/ShapeSense')
sys.path.insert(0, os.getcwd())

# Stage 4 saves each model as TorchScript (torch.jit.script), which Stages 5 and 6 load back. PyTorch now
# prints a deprecation warning for TorchScript; it still works, and switching formats would change the saved
# models. Hide only that warning, here and in the subprocesses the stages and scripts start.
import warnings
warnings.filterwarnings('ignore', message=r'`torch\.jit')
os.environ['PYTHONWARNINGS'] = 'ignore:`torch.jit'

# Colab: match the processes that prepare training batches to this machine's CPU cores.
# The config's 7 suits the original machine; on Colab it can overload a session with few cores or underuse one with many.
# Results do not depend on it (configs/pipeline.yaml, train.prep_workers), so only the speed changes.
if os.path.isdir('/content/ShapeSense'):
    import re
    workers = max(1, (os.cpu_count() or 2) - 1)
    text = open('configs/pipeline.yaml').read()
    open('configs/pipeline.yaml', 'w').write(
        re.sub(r'(?m)^(\s*prep_workers:\s*)\d+', rf'\g<1>{workers}', text, count=1))
    print('CPU cores:', os.cpu_count(), '| batch-preparing processes:', workers)
import run_pipeline as rp
from src.common import status as st
from src.common.config import load_config, resolve

CONFIG = load_config(resolve('configs/pipeline.yaml'))
ARTIFACTS = resolve(CONFIG['paths']['artifacts_dir'])
rp.restore_frozen_winners(ARTIFACTS, resolve(rp.FROZEN_WINNERS))

def run(*stages):
    """Guards, then the given stages in order, without Stage 0."""
    os.environ[st.RUN_ID_ENV] = st.new_run_id()
    guards = rp.run_guards(CONFIG)
    if not all(guards.values()):
        raise RuntimeError(f'guards failed: {guards}')
    outcome = rp.run_stages(CONFIG, list(stages), force=False)
    for n, result in outcome.items():
        if result not in ('passed', 'cached'):
            s = st.read_status(f'stage{n}', ARTIFACTS) or {}
            print(f'Stage {n} {result}: checks={s.get("checks")} error={s.get("error")}')
            raise RuntimeError(f'Stage {n} did not pass')
    print('outcome:', outcome)


## 4. Stage 1: data freeze and splits

Builds the unified dataset, applies the frozen cleaning rule (`exclude_source_labels` in `configs/pipeline.yaml`: Quick, Draw! diamonds), removes exact duplicates and blank images, and writes the group-aware splits. The raw baseline in section 8 is rebuilt from the raw files by replaying this step.

*Approximate time: about 15 seconds (Stage 1 took 11 s on the original machine).*


In [ ]:
run(1)


## 5. Stage 2: augmentation (transformation)

Class-safe augmentation. F1 after this step: see section 8.

*Approximate time: a few seconds (Stage 2 took 1 s on the original machine).*


In [ ]:
run(2)


## 6. Stage 3: synthetic generator

*Approximate time: a few seconds (Stage 3 took 5 s on the original machine).*


In [ ]:
run(3)


## 7. Stage 4 and 5: training and test evaluation

Training is per arm and per seed. Test is read once per run (see `artifacts/test_access.log`).

*Approximate time: about 2 hours on the original machine (Stage 4 took 1 h 12 min, Stage 5 took 47 min); expect longer on a T4. Keep the tab open.*

Each stage prints `Stage N: passed` when it finishes; the cell ends with `outcome: {4: 'passed', 5: 'passed'}`. A `torch.jit` deprecation warning above those lines is expected (see the notes at the top) and does not affect the results.


In [ ]:
run(4, 5)


## 8. Baseline and improvement after each data preparation step

Every row trains the same CNN (same settings, 49 steps x 60 epochs, seeds 0-4) and is scored by **macro F1 on the same 2,776 test images**. Only the data preparation changes. `scripts/improvement_table.py` documents each row.

1. `replay` rebuilds every image from its raw file, only decoded and resized to 64x64.
2. `train` trains the baseline (raw, uncleaned) and the cleaning-only model.
3. `fs` picks the input resolution on validation (feature selection).
4. `evaluate` reads the test split once, through Stage 5's logged reader.

*Approximate time: about 45 minutes on the original machine: replay a few minutes, training about 16 minutes, feature selection about 20 minutes, evaluation a few minutes.*


In [ ]:
import os
if os.path.isdir('/content/ShapeSense'):  # Colab: a restart puts us back in /content
    os.chdir('/content/ShapeSense')
!python -m scripts.improvement_table replay
!python -m scripts.improvement_table train
!python -m scripts.improvement_table fs
!python -m scripts.improvement_table evaluate

import json
table = json.load(open('results/improvement_table.json'))
rows = table['rows']
def show(i):
    r = rows[i]
    print(r['step'])
    print(f"  test macro F1 {r['mean']:.2f} +/- {r['std']:.2f} (5 seeds), delta vs baseline {r['delta_vs_baseline']:+.2f}")


### 8.1 Baseline: the collected data before any preparation

*Approximate time: instant.*


In [ ]:
show(0)

### 8.2 After cleaning

*Approximate time: instant.*


In [ ]:
show(1)

### 8.3 After transformation (standardization, then augmentation + synthetic shapes)

*Approximate time: instant.*


In [ ]:
show(2); show(3)

### 8.4 After feature selection

*Approximate time: instant.*


In [ ]:
import json
show(4)
print(json.dumps(table['feature_selection'], indent=2))

## 9. Stage 6: report

*Approximate time: about 10 seconds (Stage 6 took 9 s on the original machine).*


In [ ]:
run(6)


## 10. Final model vs. direct LLM prediction on the same test images

200 test images (20 per class, seed 0, `results/direct_llm_prediction/sample_ids.csv`) were shown blind to Claude Opus 5.5 and Gemini 4 Neon. The trained CNN (A2_0) is scored on the same images with the same metric. The baseline model's score on these 200 images is printed too.

*Approximate time: a few seconds.*


In [ ]:
import os
if os.path.isdir('/content/ShapeSense'):  # Colab: a restart puts us back in /content
    os.chdir('/content/ShapeSense')
import json
import csv, numpy as np
from sklearn.metrics import f1_score
def macro(path):
    rows = list(csv.DictReader(open(path)))
    return 100 * f1_score([r['true_label'] for r in rows], [r['predicted_label'] for r in rows], average='macro')
for name in ['trained_cnn', 'claude_opus', 'gemini_4_neon']:
    print(f"{name:14s} macro F1 {macro(f'results/direct_llm_prediction/{name}_predictions.csv'):.2f}")
sub = json.load(open('results/improvement_table.json'))['direct_llm_subset']
print(f"baseline (raw) macro F1 {sub['baseline_0']:.2f}; final A2_0 recomputed {sub['final_A2_0']:.2f}")
